# FitNova v5.1 — SSL Pretrain (rebuild, reads v5_1_dataset.zip)

Use this only if `ssl_encoder.weights.h5` is missing from your Drive
`MyDrive/fitnova_v5_results/` folder. If it's there, skip this and go
straight to `v5_1_supervised_train.ipynb`.

This notebook is functionally identical to v5.0's SSL pretrain — it just
reads the v5.1 dataset and filters to clean samples (the SSL distribution
should match what supervised will see at clean inputs).

**Cost:** ~25–45 min on L4. ~0.7 Colab compute units.

**Inputs (MyDrive/fitnova_v5/):**
- `fitnova_v5_1_src.zip`
- `v5_1_dataset.zip`

**Outputs (MyDrive/fitnova_v5_results/):**
- `ssl_encoder.weights.h5`  ← what `v5_1_supervised_train.ipynb` reads
- `ssl_history.json`
- `ssl_checkpoints/epoch_NN.weights.h5` (rolling 3 latest, for crash-resume)


In [ ]:
# ── GPU + Drive mount ────────────────────────────────────────────────────────
import os, sys, json, time, shutil
import tensorflow as tf
print("TF:    ", tf.__version__)
print("Keras: ", tf.keras.__version__)
print("GPUs:  ", tf.config.list_physical_devices("GPU"))

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE  = "/content/drive/MyDrive/fitnova_v5"
RESULTS_DIR = "/content/drive/MyDrive/fitnova_v5_results"
os.makedirs(DRIVE_BASE,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("DRIVE_BASE :", DRIVE_BASE)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
# ── Extract v5.1 backend source from Drive ──────────────────────────────────
SRC_ZIP  = f"{DRIVE_BASE}/fitnova_v5_1_src.zip"
WORK_DIR = "/content/fitnova_v5_1"

assert os.path.isfile(SRC_ZIP), (
    f"Source zip not found at {SRC_ZIP}.\n"
    f"Build it locally: python _build_v5_1_colab_bundle.py\n"
    f"Then upload fitnova_v5_1_src.zip to MyDrive/fitnova_v5/"
)

# Always re-extract so we pick up code changes.
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(WORK_DIR, exist_ok=True)
!unzip -q "$SRC_ZIP" -d "$WORK_DIR"

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print("Source ready at", WORK_DIR)
print("  has st_gcn:                 ", os.path.exists(f"{WORK_DIR}/backend/training/models/st_gcn.py"))
print("  has synthetic_perturbation: ", os.path.exists(f"{WORK_DIR}/backend/training/preprocessing/synthetic_perturbation.py"))
print("  has dataset_builder_v5_1:   ", os.path.exists(f"{WORK_DIR}/backend/training/preprocessing/dataset_builder_v5_1.py"))


In [ ]:
# ── Extract v5.1 pre-built dataset from Drive ───────────────────────────────
DATA_ZIP = f"{DRIVE_BASE}/v5_1_dataset.zip"
DATA_DIR = "/content/v5_1_dataset"

assert os.path.isfile(DATA_ZIP), (
    f"Dataset zip not found at {DATA_ZIP}.\n"
    f"Build it locally: python _build_v5_1_colab_bundle.py\n"
    f"Then upload v5_1_dataset.zip to MyDrive/fitnova_v5/"
)

shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)
!unzip -q "$DATA_ZIP" -d "$DATA_DIR"

print("Dataset files:", sorted(os.listdir(DATA_DIR)))

with open(f"{DATA_DIR}/dataset_info.json") as f:
    DATASET_INFO = json.load(f)
for k in ("version", "label_strategy", "n_exercises", "n_joint_groups",
          "target_frames", "n_canonical_joints", "n_angular",
          "perturbations_per_rep_train", "severity_to_quality"):
    print(f"  {k:<32s}: {DATASET_INFO[k]}")


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
from tensorflow import keras
from backend.training.models.st_gcn import build_v5_ssl_model

SSL_DIR     = f"{RESULTS_DIR}/ssl_checkpoints"
SSL_HISTORY = f"{RESULTS_DIR}/ssl_history.json"
SSL_FINAL   = f"{RESULTS_DIR}/ssl_encoder.weights.h5"
os.makedirs(SSL_DIR, exist_ok=True)


In [ ]:
# ── Hyperparameters (same as v5.0 SSL) ───────────────────────────────────────
EPOCHS     = 20
BATCH_SIZE = 32
MASK_RATIO = 0.30
LR         = 1e-3
SEED       = 42

T_FRAMES = DATASET_INFO["target_frames"]
J        = DATASET_INFO["n_canonical_joints"]


In [ ]:
# ── Load training pose (clean samples only) ──────────────────────────────────
train_npz = np.load(f"{DATA_DIR}/train.npz")
val_npz   = np.load(f"{DATA_DIR}/val.npz")

train_clean_mask = train_npz["perturb_severity_idx"] == 0
val_clean_mask   = val_npz["perturb_severity_idx"] == 0
train_pose = train_npz["pose"][train_clean_mask].astype(np.float32)
val_pose   = val_npz["pose"][val_clean_mask].astype(np.float32)

print(f"train_pose: {train_pose.shape}  (clean only out of {len(train_npz['pose'])} total)")
print(f"val_pose:   {val_pose.shape}    (clean only out of {len(val_npz['pose'])} total)")
print(f"finite:     {np.isfinite(train_pose).all()}, {np.isfinite(val_pose).all()}")


In [ ]:
# ── Build masked-joint dataset (random per-joint per-frame mask) ─────────────
def make_ssl_dataset(pose, mask_ratio, batch_size, shuffle, seed):
    pose = pose.astype(np.float32)
    pose_xyz = pose[..., :3]
    n = len(pose)

    def gen():
        rng = np.random.default_rng(seed)
        order = np.arange(n)
        while True:
            if shuffle:
                rng.shuffle(order)
            for i in range(0, n, batch_size):
                idx  = order[i:i+batch_size]
                p    = pose[idx].copy()
                mask = (rng.random((*p.shape[:3], 1)) < mask_ratio).astype(np.float32)
                p_masked = p.copy()
                p_masked[..., :3] *= (1.0 - mask)
                yield (
                    {"pose_masked": p_masked, "mask": mask},
                    pose_xyz[idx],
                )

    sig = (
        {
            "pose_masked": tf.TensorSpec(shape=(None, T_FRAMES, J, 4), dtype=tf.float32),
            "mask":        tf.TensorSpec(shape=(None, T_FRAMES, J, 1), dtype=tf.float32),
        },
        tf.TensorSpec(shape=(None, T_FRAMES, J, 3), dtype=tf.float32),
    )
    return tf.data.Dataset.from_generator(gen, output_signature=sig).prefetch(2)

steps_per_epoch  = max(1, len(train_pose) // BATCH_SIZE)
val_steps        = max(1, len(val_pose)   // BATCH_SIZE)
print(f"steps/epoch: {steps_per_epoch}   val steps: {val_steps}")

train_ds = make_ssl_dataset(train_pose, MASK_RATIO, BATCH_SIZE, shuffle=True,  seed=SEED)
val_ds   = make_ssl_dataset(val_pose,   MASK_RATIO, BATCH_SIZE, shuffle=False, seed=SEED + 1)


In [ ]:
# ── Build model ──────────────────────────────────────────────────────────────
def masked_recon_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

ssl_model = build_v5_ssl_model(
    target_frames=T_FRAMES,
    n_joints=J,
    n_pose_channels=4,
)
ssl_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LR, weight_decay=1e-4, clipnorm=1.0),
    loss={"pose_recon": masked_recon_loss},
)
ssl_model.summary(line_length=120)


In [ ]:
# ── Crash-resume detection ──────────────────────────────────────────────────
def find_latest_epoch(ckpt_dir):
    if not os.path.isdir(ckpt_dir):
        return 0
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("epoch_") and f.endswith(".weights.h5")]
    if not files:
        return 0
    return max(int(f.split("_")[1].split(".")[0]) for f in files)

initial_epoch = find_latest_epoch(SSL_DIR)
if initial_epoch > 0:
    last_ckpt = f"{SSL_DIR}/epoch_{initial_epoch:02d}.weights.h5"
    print(f"Resuming from epoch {initial_epoch} ({last_ckpt})")
    ssl_model.load_weights(last_ckpt)
else:
    print("Fresh start (no prior SSL checkpoints found)")

history_acc = {"loss": [], "val_loss": []}
if os.path.isfile(SSL_HISTORY):
    with open(SSL_HISTORY) as f:
        history_acc = json.load(f)
    history_acc.setdefault("loss", [])
    history_acc.setdefault("val_loss", [])
    print(f"Loaded prior history: {len(history_acc['loss'])} epochs")


In [ ]:
# ── Per-epoch checkpoint callback ───────────────────────────────────────────
class DriveCheckpoint(keras.callbacks.Callback):
    def __init__(self, ckpt_dir, history_path, history_acc, total_epochs):
        super().__init__()
        self.ckpt_dir = ckpt_dir
        self.history_path = history_path
        self.history_acc  = history_acc
        self.total_epochs = total_epochs

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        ep1 = epoch + 1
        ckpt = f"{self.ckpt_dir}/epoch_{ep1:02d}.weights.h5"
        self.model.save_weights(ckpt)
        self.history_acc["loss"].append(float(logs.get("loss", 0.0)))
        self.history_acc["val_loss"].append(float(logs.get("val_loss", 0.0)))
        with open(self.history_path, "w") as f:
            json.dump(self.history_acc, f, indent=2)
        files = sorted(
            f for f in os.listdir(self.ckpt_dir)
            if f.startswith("epoch_") and f.endswith(".weights.h5")
        )
        for f in files[:-3]:
            try:
                os.remove(os.path.join(self.ckpt_dir, f))
            except OSError:
                pass
        print(f"  [saved {os.path.basename(ckpt)}]")


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
if initial_epoch >= EPOCHS:
    print(f"Already trained for {initial_epoch} epochs — nothing to do. Skip to next cell.")
else:
    callbacks = [DriveCheckpoint(SSL_DIR, SSL_HISTORY, history_acc, EPOCHS)]
    ssl_model.fit(
        train_ds,
        validation_data=val_ds,
        steps_per_epoch=steps_per_epoch,
        validation_steps=val_steps,
        epochs=EPOCHS,
        initial_epoch=initial_epoch,
        callbacks=callbacks,
        verbose=2,
    )


In [ ]:
# ── Save final weights and summarise ─────────────────────────────────────────
ssl_model.save_weights(SSL_FINAL)
print(f"Saved final SSL encoder to {SSL_FINAL}")
print(f"  size: {os.path.getsize(SSL_FINAL) / 1e6:.2f} MB")

with open(SSL_HISTORY) as f:
    h = json.load(f)
print(f"\nFinal SSL stats over {len(h['loss'])} epochs:")
print(f"  train loss: first={h['loss'][0]:.6f}  last={h['loss'][-1]:.6f}  best={min(h['loss']):.6f}")
print(f"  val   loss: first={h['val_loss'][0]:.6f}  last={h['val_loss'][-1]:.6f}  best={min(h['val_loss']):.6f}")
print()
print("Done. Now go run v5_1_supervised_train.ipynb.")
